In [1]:
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Bidirectional, GRU, Dense, 
Dropout, Attention, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [2]:
fake = pd.read_csv('Fake.csv')  # Fake News
true = pd.read_csv('True.csv')  # True News

In [3]:
fake.head(5)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
true.head(5)

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [5]:
fake['label'] = 1
true['label'] = 0

In [6]:
df = pd.concat([fake,true], axis = 0)

In [7]:
df.sample(frac=1, random_state=42)  # Shuffling rows to balance data in train test split

,title,text,subject,date,label
22216,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1
4436,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0
1526,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0
1377,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",1
8995,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",0
...,...,...,...,...,...
11284,UNREAL! CBS’S TED KOPPEL Tells Sean Hannity He...,,politics,"Mar 27, 2017",1
21251,PM May seeks to ease Japan's Brexit fears duri...,LONDON/TOKYO (Reuters) - British Prime Ministe...,worldnews,"August 29, 2017",0
14677,Merkel: Difficult German coalition talks can r...,BERLIN (Reuters) - Chancellor Angela Merkel sa...,worldnews,"November 16, 2017",0
860,Trump Stole An Idea From North Korean Propaga...,Jesus f*cking Christ our President* is a moron...,News,"July 14, 2017",1


In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'https\S+',"", text)  # To Remove https links
    text = re.sub(r'[^a-zA-Z]', " ", text)  # to get only alphabets and space
    text = re.sub(r"\s+", " ", text)  # Punctuation marks
    return text.strip()

In [9]:
'      peeyush choudhary      '.strip()

'peeyush choudhary'

In [10]:
df['content'] = df['text']+' '+df['title']

In [11]:
df['content'] = df['content'].apply(lambda x: clean_text(x))

In [12]:
df

,title,text,subject,date,label,content
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1,donald trump just couldn t wish all americans ...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1,house intelligence committee chairman devin nu...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1,on friday it was revealed that former milwauke...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1,on christmas day donald trump announced that h...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1,pope francis used his annual christmas day mes...
...,...,...,...,...,...,...
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",0,brussels reuters nato allies on tuesday welcom...
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",0,london reuters lexisnexis a provider of legal ...
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",0,minsk reuters in the shadow of disused soviet ...
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",0,moscow reuters vatican secretary of state card...


# Tokenization

In [13]:
max_words = 50000
max_len = 500

In [14]:
tokenizer = Tokenizer(num_words=max_words, oov_token = '<OOV>')

In [15]:
tokenizer.fit_on_texts(df['content'])

In [16]:
len(tokenizer.word_index)

111970

In [17]:
X = tokenizer.texts_to_sequences(df['content'])

In [18]:
max([max(i) for i in X if len(i)>1])

49999

In [19]:
X = pad_sequences(X, 
                  maxlen = max_len, 
                  padding = 'post', 
                  truncating = 'post')

In [20]:
y = df['label']

In [21]:
y

0        1
1        1
2        1
3        1
4        1
        ..
21412    0
21413    0
21414    0
21415    0
21416    0
Name: label, Length: 44898, dtype: int64

In [22]:
x_train, x_test, y_train, y_test = train_test_split(X,y,
                                                   test_size = 0.25,
                                                   random_state =42,
                                                   stratify = y)

# Model

In [23]:
inputs = Input(
    shape = (max_len,)
)

embedding = Embedding(
    input_dim = max_words,
    output_dim = 128)(inputs)

gru1 = Bidirectional(
    GRU(128, return_sequences = True))(embedding)

gru2 = Bidirectional(
    GRU(64, return_sequences = True))(gru1)

attention = Attention()(
    [gru2,gru2])

pool = GlobalAveragePooling1D()(attention)

dense = Dense(64, activation = 'relu')(pool)

drop = Dropout(0.3)(dense)

output = Dense(1, activation = 'sigmoid')(drop)

model = Model(inputs, output)

In [24]:
model.compile(optimizer= 'adam', loss = 'binary_crossentropy', metrics = ['accuracy']) 

In [25]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 500)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 500, 128)          │       6,400,000 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional (Bidirectional) │ (None, 500, 256)          │         198,144 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional_1               │ (None, 500, 128)          │         123,648 │ bidirectional[0][0]        │
│ (Bidirectional)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention (Attention)         │ (None, 500, 128)          │               0 │ bidirectional_1[0][0],     │
│                               │                           │                 │ bidirectional_1[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d      │ (None, 128)               │               0 │ attention[0][0]            │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 64)                │           8,256 │ global_average_pooling1d[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 64)                │               0 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 1)                 │              65 │ dropout[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 6,730,113 (25.67 MB)

 Trainable params: 6,730,113 (25.67 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
checkpoint = ModelCheckpoint('models/bigru_attention_model.h5',
                            save_best_only = True,
                            monitor = 'val_accuracy',
                            mode = 'max')

In [27]:
earlystopping = EarlyStopping(
    patience = 3,
    restore_best_weights = True
)

In [28]:
history = model.fit(x_train, y_train, validation_split = 0.2, epochs=2, batch_size = 64,
                   callbacks=[checkpoint, earlystopping])

Epoch 1/2
421/421 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8200 - loss: 0.3214

421/421 ━━━━━━━━━━━━━━━━━━━━ 1000s 2s/step - accuracy: 0.9270 - loss: 0.1564 - val_accuracy: 0.9926 - val_loss: 0.0224
Epoch 2/2
421/421 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9968 - loss: 0.0135

421/421 ━━━━━━━━━━━━━━━━━━━━ 999s 2s/step - accuracy: 0.9974 - loss: 0.0101 - val_accuracy: 0.9972 - val_loss: 0.0086
